In [ ]:

from __future__ import annotations

import random
from dataclasses import dataclass
from typing import Dict, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
import scipy.sparse as sp
from sklearn.impute import KNNImputer, SimpleImputer
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import NearestNeighbors

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler


SEED = 42


def set_seed(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


@dataclass
class SplitPack:
    train_idx: np.ndarray
    valid_idx: np.ndarray
    test_idx: np.ndarray


def stratified_811_split(df: pd.DataFrame, label_col: str, seed: int = SEED) -> SplitPack:
    idx = np.arange(len(df))
    y = df[label_col].astype(str).values
    train_idx, temp_idx = train_test_split(
        idx, test_size=0.2, random_state=seed, stratify=y
    )
    y_temp = df.iloc[temp_idx][label_col].astype(str).values
    valid_idx, test_idx = train_test_split(
        temp_idx, test_size=0.5, random_state=seed, stratify=y_temp
    )
    return SplitPack(train_idx=np.array(train_idx), valid_idx=np.array(valid_idx), test_idx=np.array(test_idx))


class MixedTabularPreprocessor:
    """Fit min-max on numeric train columns, one-hot on categoricals, preserving NaN mask for numerics."""

    def __init__(self, numeric_cols: Sequence[str], categorical_cols: Sequence[str]):
        self.numeric_cols = list(numeric_cols)
        self.categorical_cols = list(categorical_cols)
        self.scalers: Dict[str, Tuple[float, float]] = {}
        self.cat_levels: Dict[str, List[str]] = {}
        self.feature_names_: List[str] = []

    def fit(self, df: pd.DataFrame) -> "MixedTabularPreprocessor":
        self.feature_names_ = []
        for c in self.numeric_cols:
            s = pd.to_numeric(df[c], errors="coerce")
            finite = s[np.isfinite(s)]
            if len(finite) == 0:
                mn, mx = 0.0, 1.0
            else:
                mn = float(finite.min())
                mx = float(finite.max())
                if mx == mn:
                    mx = mn + 1.0
            self.scalers[c] = (mn, mx)
            self.feature_names_.append(c)
        for c in self.categorical_cols:
            lv = sorted(df[c].fillna("__MISSING__").astype(str).unique().tolist())
            self.cat_levels[c] = lv
            self.feature_names_.extend([f"{c}__{x}" for x in lv])
        return self

    def transform(self, df: pd.DataFrame) -> Tuple[np.ndarray, np.ndarray, List[str]]:
        numeric_parts = []
        numeric_mask = []
        for c in self.numeric_cols:
            s = pd.to_numeric(df[c], errors="coerce").astype(float)
            mn, mx = self.scalers[c]
            arr = s.values.astype(np.float32)
            mask = np.isfinite(arr).astype(np.float32)
            arr = (arr - mn) / (mx - mn)
            arr = np.clip(arr, 0.0, 1.0)
            arr[~np.isfinite(arr)] = np.nan
            numeric_parts.append(arr[:, None])
            numeric_mask.append(mask[:, None])
        X_num = np.hstack(numeric_parts) if numeric_parts else np.empty((len(df), 0), dtype=np.float32)
        M_num = np.hstack(numeric_mask) if numeric_mask else np.empty((len(df), 0), dtype=np.float32)

        cat_parts = []
        cat_mask = []
        for c in self.categorical_cols:
            levels = self.cat_levels[c]
            s = df[c].fillna("__MISSING__").astype(str)
            arr = np.zeros((len(df), len(levels)), dtype=np.float32)
            level_index = {v: i for i, v in enumerate(levels)}
            default_idx = level_index.get("__MISSING__", 0)
            for i, val in enumerate(s.values):
                arr[i, level_index.get(val, default_idx)] = 1.0
            cat_parts.append(arr)
            cat_mask.append(np.ones_like(arr, dtype=np.float32))
        X_cat = np.hstack(cat_parts) if cat_parts else np.empty((len(df), 0), dtype=np.float32)
        M_cat = np.hstack(cat_mask) if cat_mask else np.empty((len(df), 0), dtype=np.float32)

        X = np.hstack([X_num, X_cat]).astype(np.float32)
        M = np.hstack([M_num, M_cat]).astype(np.float32)
        return X, M, list(self.feature_names_)


def _limited_group_edges(ids: np.ndarray, max_group_neighbors: int = 20) -> List[Tuple[int, int]]:
    edges: List[Tuple[int, int]] = []
    m = len(ids)
    if m <= 1:
        return edges
    half = max(1, max_group_neighbors // 2)
    for i, src in enumerate(ids):
        left = ids[max(0, i - half): i]
        right = ids[i + 1: i + 1 + half]
        for dst in np.concatenate([left, right]):
            if int(src) != int(dst):
                edges.append((int(src), int(dst)))
    return edges


def build_sparse_graph(
    meta_df: pd.DataFrame,
    X_filled: np.ndarray,
    group_cols: Sequence[str],
    tau: float = 0.55,
    top_k: int = 15,
    max_group_neighbors: int = 20,
) -> sp.coo_matrix:
    meta_df = meta_df.reset_index(drop=True).copy()
    n = len(meta_df)
    if X_filled.shape[0] != n:
        raise ValueError(f"X_filled rows {X_filled.shape[0]} != meta rows {n}")

    meta_df["_node_id"] = np.arange(n, dtype=np.int64)
    edges: List[Tuple[int, int]] = []

    for col in group_cols:
        if col not in meta_df.columns:
            continue
        tmp = meta_df[["_node_id", col]].copy()
        tmp[col] = tmp[col].fillna("__MISSING__").astype(str)
        for _, grp in tmp.groupby(col, sort=False):
            ids = grp["_node_id"].to_numpy(dtype=np.int64)
            edges.extend(_limited_group_edges(ids, max_group_neighbors=max_group_neighbors))

    if n > 1 and top_k > 0:
        nn_k = min(top_k + 1, n)
        nn_model = NearestNeighbors(metric="cosine", algorithm="brute", n_neighbors=nn_k)
        nn_model.fit(X_filled)
        dists, nbrs = nn_model.kneighbors(X_filled)
        sims = 1.0 - dists
        for i in range(n):
            for sim, j in zip(sims[i, 1:], nbrs[i, 1:]):
                if sim >= tau and i != int(j):
                    edges.append((int(i), int(j)))

    if not edges:
        rows = np.arange(n)
        cols = np.arange(n)
        data = np.ones(n, dtype=np.float32)
        return sp.coo_matrix((data, (rows, cols)), shape=(n, n))

    rows, cols = zip(*edges)
    rows = np.asarray(rows, dtype=np.int64)
    cols = np.asarray(cols, dtype=np.int64)

    if rows.max(initial=0) >= n or cols.max(initial=0) >= n or rows.min(initial=0) < 0 or cols.min(initial=0) < 0:
        raise ValueError(f"Invalid graph edge index: row[{rows.min()}..{rows.max()}], col[{cols.min()}..{cols.max()}], n={n}")

    data = np.ones(len(rows), dtype=np.float32)
    adj = sp.coo_matrix((data, (rows, cols)), shape=(n, n))
    adj = adj + adj.T
    adj = adj.maximum(adj.T)
    adj = adj + sp.eye(n, dtype=np.float32)

    deg = np.array(adj.sum(1)).flatten()
    deg_inv_sqrt = np.power(np.clip(deg, 1.0, None), -0.5)
    d_inv_sqrt = sp.diags(deg_inv_sqrt)
    norm_adj = d_inv_sqrt @ adj @ d_inv_sqrt
    return norm_adj.tocoo()


def scipy_to_torch_sparse(mat: sp.coo_matrix, device: torch.device) -> torch.Tensor:
    mat = mat.tocoo().astype(np.float32)
    idx = torch.from_numpy(np.vstack([mat.row, mat.col]).astype(np.int64))
    vals = torch.from_numpy(mat.data)
    return torch.sparse_coo_tensor(idx, vals, torch.Size(mat.shape), device=device).coalesce()


class SparseGCNLayer(nn.Module):
    def __init__(self, in_dim: int, out_dim: int):
        super().__init__()
        self.weight = nn.Parameter(torch.empty(in_dim, out_dim))
        self.bias = nn.Parameter(torch.zeros(out_dim))
        nn.init.xavier_uniform_(self.weight)

    def forward(self, x: torch.Tensor, adj: torch.Tensor) -> torch.Tensor:
        h = torch.sparse.mm(adj, x)
        return h @ self.weight + self.bias


class GCNGenerator(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int = 128, dropout: float = 0.3):
        super().__init__()
        self.gcn1 = SparseGCNLayer(input_dim, hidden_dim)
        self.gcn2 = SparseGCNLayer(hidden_dim, hidden_dim)
        self.norm1 = nn.LayerNorm(hidden_dim)
        self.norm2 = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(dropout)
        self.decoder = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, input_dim),
        )

    def forward(self, x: torch.Tensor, adj: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        h1 = torch.relu(self.norm1(self.gcn1(x, adj)))
        h1 = self.dropout(h1)
        h2 = torch.relu(self.norm2(self.gcn2(h1, adj) + h1))
        out = self.decoder(h2)
        return out, h2


class Critic(nn.Module):
    def __init__(self, input_dim: int, hidden: int = 128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden),
            nn.LeakyReLU(0.2),
            nn.Linear(hidden, hidden // 2),
            nn.LeakyReLU(0.2),
            nn.Linear(hidden // 2, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x).squeeze(-1)


def gradient_penalty(critic: Critic, real_x: torch.Tensor, fake_x: torch.Tensor, device: torch.device) -> torch.Tensor:
    alpha = torch.rand(real_x.size(0), 1, device=device)
    interpolated = alpha * real_x + (1 - alpha) * fake_x
    interpolated.requires_grad_(True)
    prob = critic(interpolated)
    grads = torch.autograd.grad(
        outputs=prob, inputs=interpolated, grad_outputs=torch.ones_like(prob),
        create_graph=True, retain_graph=True, only_inputs=True
    )[0]
    norm = grads.norm(2, dim=1)
    return ((norm - 1.0) ** 2).mean()


@dataclass
class GraphImputeResult:
    generator_state: Dict[str, torch.Tensor]
    feature_names: List[str]
    preprocessor: MixedTabularPreprocessor
    train_columns: List[str]
    numeric_cols: List[str]
    categorical_cols: List[str]
    tau: float
    top_k: int
    history: pd.DataFrame


def train_graph_imputer(
    train_meta: pd.DataFrame,
    train_df: pd.DataFrame,
    numeric_cols: Sequence[str],
    categorical_cols: Sequence[str],
    group_cols: Sequence[str],
    hidden_dim: int = 128,
    dropout: float = 0.3,
    lr: float = 1e-2,
    weight_decay: float = 5e-5,
    epochs: int = 100,
    patience: int = 10,
    tau: float = 0.55,
    top_k: int = 15,
    critic_steps: int = 2,
    lambda_gp: float = 10.0,
    use_adversarial: bool = True,
    device: Optional[str] = None,
) -> GraphImputeResult:
    set_seed()
    dev = torch.device(device or ("cuda" if torch.cuda.is_available() else "cpu"))

    train_meta = train_meta.reset_index(drop=True).copy()
    train_df = train_df.reset_index(drop=True).copy()

    pre = MixedTabularPreprocessor(numeric_cols=numeric_cols, categorical_cols=categorical_cols).fit(train_df)
    X_nan, M, _ = pre.transform(train_df)
    X_in = np.nan_to_num(X_nan, nan=0.0).astype(np.float32)
    adj = build_sparse_graph(train_meta, X_in, group_cols=group_cols, tau=tau, top_k=top_k)

    x = torch.tensor(X_in, dtype=torch.float32, device=dev)
    mask = torch.tensor(M, dtype=torch.float32, device=dev)
    x_target = torch.tensor(np.nan_to_num(X_nan, nan=0.0), dtype=torch.float32, device=dev)
    adj_t = scipy_to_torch_sparse(adj, dev)

    gen = GCNGenerator(input_dim=x.shape[1], hidden_dim=hidden_dim, dropout=dropout).to(dev)
    critic = Critic(input_dim=x.shape[1], hidden=hidden_dim).to(dev)

    opt_g = torch.optim.Adam(gen.parameters(), lr=lr, weight_decay=weight_decay)
    opt_c = torch.optim.Adam(critic.parameters(), lr=lr, weight_decay=weight_decay)

    best_loss = float("inf")
    best_state = None
    wait = 0
    rows = []

    for epoch in range(1, epochs + 1):
        gen.train()
        critic.train()

        recon, _ = gen(x, adj_t)
        recon_loss = (((recon - x_target) ** 2) * mask).sum() / mask.sum().clamp_min(1.0)

        c_loss = torch.tensor(0.0, device=dev)
        if use_adversarial:
            for _ in range(critic_steps):
                opt_c.zero_grad(set_to_none=True)
                with torch.no_grad():
                    fake, _ = gen(x, adj_t)
                real_score = critic(x_target)
                fake_score = critic(fake)
                gp = gradient_penalty(critic, x_target.detach(), fake.detach(), dev)
                c_loss = fake_score.mean() - real_score.mean() + lambda_gp * gp
                c_loss.backward()
                opt_c.step()

        opt_g.zero_grad(set_to_none=True)
        recon, _ = gen(x, adj_t)
        recon_loss = (((recon - x_target) ** 2) * mask).sum() / mask.sum().clamp_min(1.0)
        g_loss = recon_loss
        adv_term = torch.tensor(0.0, device=dev)
        if use_adversarial:
            adv_term = -critic(recon).mean()
            g_loss = recon_loss + 0.1 * adv_term
        g_loss.backward()
        nn.utils.clip_grad_norm_(gen.parameters(), 1.0)
        opt_g.step()

        total = float(g_loss.detach().cpu())
        rows.append({
            "epoch": epoch,
            "recon_loss": float(recon_loss.detach().cpu()),
            "critic_loss": float(c_loss.detach().cpu()),
            "generator_loss": float(g_loss.detach().cpu()),
            "adv_term": float(adv_term.detach().cpu()) if use_adversarial else 0.0,
        })

        if total < best_loss - 1e-5:
            best_loss = total
            best_state = {k: v.detach().cpu().clone() for k, v in gen.state_dict().items()}
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                break

    if best_state is None:
        best_state = {k: v.detach().cpu().clone() for k, v in gen.state_dict().items()}

    return GraphImputeResult(
        generator_state=best_state,
        feature_names=list(pre.feature_names_),
        preprocessor=pre,
        train_columns=list(train_df.columns),
        numeric_cols=list(numeric_cols),
        categorical_cols=list(categorical_cols),
        tau=tau,
        top_k=top_k,
        history=pd.DataFrame(rows),
    )


def apply_graph_imputer(
    fit_result: GraphImputeResult,
    meta_df: pd.DataFrame,
    data_df: pd.DataFrame,
    group_cols: Sequence[str],
    hidden_dim: int = 128,
    dropout: float = 0.3,
    device: Optional[str] = None,
) -> pd.DataFrame:
    dev = torch.device(device or ("cuda" if torch.cuda.is_available() else "cpu"))
    meta_df = meta_df.reset_index(drop=True).copy()
    data_df = data_df.reset_index(drop=True).copy()

    pre = fit_result.preprocessor
    X_nan, M, _ = pre.transform(data_df)
    X_in = np.nan_to_num(X_nan, nan=0.0).astype(np.float32)
    adj = build_sparse_graph(meta_df, X_in, group_cols=group_cols, tau=fit_result.tau, top_k=fit_result.top_k)
    adj_t = scipy_to_torch_sparse(adj, dev)

    gen = GCNGenerator(input_dim=X_in.shape[1], hidden_dim=hidden_dim, dropout=dropout).to(dev)
    gen.load_state_dict(fit_result.generator_state)
    gen.eval()
    with torch.no_grad():
        x = torch.tensor(X_in, dtype=torch.float32, device=dev)
        recon, emb = gen(x, adj_t)
        recon_np = recon.cpu().numpy()
        emb_np = emb.cpu().numpy()

    out = pd.DataFrame(recon_np, columns=fit_result.feature_names, index=data_df.index)
    for c in fit_result.numeric_cols:
        if c in data_df.columns and c in fit_result.feature_names:
            original = pd.to_numeric(data_df[c], errors="coerce")
            col_idx = fit_result.feature_names.index(c)
            out[c] = np.where(np.isfinite(original.values), np.nan_to_num(X_nan[:, col_idx], nan=np.nan), out[c].values)
    for i in range(min(emb_np.shape[1], 16)):
        out[f"g_emb_{i:02d}"] = emb_np[:, i]
    return out


def fit_baseline_imputer(preprocessor: MixedTabularPreprocessor, train_df: pd.DataFrame, strategy: str):
    X_nan = preprocessor.transform(train_df)[0]
    if strategy == "mean":
        imputer = SimpleImputer(strategy="mean")
    elif strategy == "median":
        imputer = SimpleImputer(strategy="median")
    elif strategy == "knn":
        imputer = KNNImputer(n_neighbors=5)
    else:
        raise ValueError(strategy)
    imputer.fit(X_nan)
    return imputer


def apply_baseline_imputer(imputer, preprocessor: MixedTabularPreprocessor, df: pd.DataFrame) -> pd.DataFrame:
    X_nan = preprocessor.transform(df)[0]
    X_imp = imputer.transform(X_nan)
    return pd.DataFrame(X_imp, columns=preprocessor.feature_names_, index=df.index)


class SequenceDataset(Dataset):
    def __init__(self, seq_x: np.ndarray, static_x: np.ndarray, y: np.ndarray):
        self.seq_x = torch.tensor(seq_x, dtype=torch.float32)
        self.static_x = torch.tensor(static_x, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.seq_x[idx], self.static_x[idx], self.y[idx]


class GRUAttentionLite(nn.Module):
    def __init__(self, seq_input_dim: int, static_input_dim: int, hidden_dim: int, num_classes: int, attention: bool = True, dropout: float = 0.2):
        super().__init__()
        self.gru = nn.GRU(seq_input_dim, hidden_dim, num_layers=1, batch_first=True)
        self.attn = nn.Linear(hidden_dim, 1) if attention else None
        self.static_net = nn.Sequential(
            nn.Linear(static_input_dim, max(32, hidden_dim // 2)),
            nn.ReLU(),
            nn.Dropout(dropout)
        ) if static_input_dim > 0 else None
        fusion_dim = hidden_dim + (max(32, hidden_dim // 2) if static_input_dim > 0 else 0)
        self.head = nn.Sequential(
            nn.Linear(fusion_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes),
        )

    def forward(self, seq_x: torch.Tensor, static_x: torch.Tensor):
        out, _ = self.gru(seq_x)
        if self.attn is not None:
            alpha = torch.softmax(self.attn(out).squeeze(-1), dim=1)
            seq_repr = (out * alpha.unsqueeze(-1)).sum(dim=1)
        else:
            seq_repr = out[:, -1, :]
        if self.static_net is not None:
            st = self.static_net(static_x)
            z = torch.cat([seq_repr, st], dim=1)
        else:
            z = seq_repr
        return self.head(z)


class SimpleBiLSTM(nn.Module):
    def __init__(self, seq_input_dim: int, static_input_dim: int, hidden_dim: int, num_classes: int, dropout: float = 0.2):
        super().__init__()
        self.rnn = nn.LSTM(seq_input_dim, hidden_dim, num_layers=1, batch_first=True, bidirectional=True)
        self.static_net = nn.Sequential(
            nn.Linear(static_input_dim, max(32, hidden_dim // 2)),
            nn.ReLU(),
            nn.Dropout(dropout)
        ) if static_input_dim > 0 else None
        fusion_dim = hidden_dim * 2 + (max(32, hidden_dim // 2) if static_input_dim > 0 else 0)
        self.head = nn.Sequential(
            nn.Linear(fusion_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes),
        )

    def forward(self, seq_x: torch.Tensor, static_x: torch.Tensor):
        out, _ = self.rnn(seq_x)
        seq_repr = out[:, -1, :]
        if self.static_net is not None:
            st = self.static_net(static_x)
            z = torch.cat([seq_repr, st], dim=1)
        else:
            z = seq_repr
        return self.head(z)


class ClassBalancedFocalLoss(nn.Module):
    def __init__(self, class_counts, gamma: float = 2.0, beta: float = 0.999):
        super().__init__()
        cc = np.maximum(np.asarray(class_counts, dtype=np.float64), 1.0)
        effective_num = 1.0 - np.power(beta, cc)
        weights = (1.0 - beta) / np.clip(effective_num, 1e-12, None)
        weights = weights / weights.sum() * len(cc)
        self.register_buffer("class_weights", torch.tensor(weights, dtype=torch.float32))
        self.gamma = gamma

    def forward(self, logits: torch.Tensor, target: torch.Tensor):
        ce = nn.functional.cross_entropy(logits, target.long(), reduction='none', weight=self.class_weights)
        pt = torch.exp(-ce)
        focal = ((1.0 - pt) ** self.gamma) * ce
        return focal.mean()


class OrdinalDistancePenalty(nn.Module):
    def __init__(self, num_classes: int):
        super().__init__()
        idx = torch.arange(num_classes, dtype=torch.float32)
        self.register_buffer('class_index', idx)

    def forward(self, logits: torch.Tensor, target: torch.Tensor):
        prob = torch.softmax(logits, dim=1)
        exp_rank = (prob * self.class_index.unsqueeze(0)).sum(dim=1)
        return ((exp_rank - target.float()) ** 2).mean()


@dataclass
class FitResult:
    model: nn.Module
    history: pd.DataFrame
    best_threshold: float
    best_metric_name: str


def build_model(model_type: str, seq_input_dim: int, static_input_dim: int, hidden_dim: int, num_classes: int):
    key = model_type.lower()
    if key in ('gru', 'gruattentionlite', 'gru_attention_lite'):
        return GRUAttentionLite(seq_input_dim, static_input_dim, hidden_dim, num_classes, attention=True)
    if key in ('bilstm', 'lstm'):
        return SimpleBiLSTM(seq_input_dim, static_input_dim, hidden_dim, num_classes)
    return GRUAttentionLite(seq_input_dim, static_input_dim, hidden_dim, num_classes, attention=False)


def _multiclass_auc(y_true: np.ndarray, prob: np.ndarray, num_classes: int):
    try:
        y_bin = np.eye(num_classes)[y_true.astype(int)]
        return float(roc_auc_score(y_bin, prob, multi_class='ovr', average='macro'))
    except Exception:
        return float('nan')


def evaluate_sequence_model_plus(model: nn.Module, ds: Dataset, num_classes: int, threshold: float = 0.5, device: Optional[str] = None) -> Dict[str, float]:
    dev = torch.device(device or ('cuda' if torch.cuda.is_available() else 'cpu'))
    model.to(dev)
    model.eval()
    loader = DataLoader(ds, batch_size=512, shuffle=False)
    y_true, y_pred, probs = [], [], []
    with torch.no_grad():
        for seq_x, static_x, y in loader:
            seq_x = seq_x.to(dev)
            static_x = static_x.to(dev)
            logits = model(seq_x, static_x)
            prob = torch.softmax(logits, dim=1)
            pred = prob.argmax(dim=1)
            y_true.extend(y.numpy().tolist())
            y_pred.extend(pred.cpu().numpy().tolist())
            probs.append(prob.cpu().numpy())
    y_true = np.asarray(y_true, dtype=int)
    y_pred = np.asarray(y_pred, dtype=int)
    prob = np.vstack(probs)
    out = {
        'accuracy': float(accuracy_score(y_true, y_pred)),
        'macro_f1': float(f1_score(y_true, y_pred, average='macro', zero_division=0)),
        'weighted_f1': float(f1_score(y_true, y_pred, average='weighted', zero_division=0)),
        'macro_precision': float(precision_score(y_true, y_pred, average='macro', zero_division=0)),
        'macro_recall': float(recall_score(y_true, y_pred, average='macro', zero_division=0)),
        'auc_ovr': _multiclass_auc(y_true, prob, num_classes),
    }
    for label_name, idx in [('label_A',0),('label_B',1),('label_C',2),('label_D',3),('label_E',4)]:
        mask = (y_true == idx).astype(int)
        pred_mask = (y_pred == idx).astype(int)
        out[f'{label_name}_f1'] = float(f1_score(mask, pred_mask, zero_division=0))
        out[f'{label_name}_precision'] = float(precision_score(mask, pred_mask, zero_division=0))
        out[f'{label_name}_recall'] = float(recall_score(mask, pred_mask, zero_division=0))
    return out


evaluate_sequence_model = evaluate_sequence_model_plus


def fit_sequence_model_plus(
    train_ds: Dataset,
    valid_ds: Dataset,
    seq_input_dim: int,
    static_input_dim: int,
    num_classes: int,
    model_type: str,
    hidden_dim: int = 64,
    lr: float = 1e-3,
    epochs: int = 40,
    patience: int = 6,
    batch_size: int = 256,
    attention: bool = True,
    loss_name: str = 'ce',
    gamma: float = 2.0,
    ordinal_lambda: float = 0.0,
    use_one_cycle: bool = True,
    device: Optional[str] = None,
) -> FitResult:
    set_seed()
    dev = torch.device(device or ('cuda' if torch.cuda.is_available() else 'cpu'))
    model = build_model(model_type, seq_input_dim, static_input_dim, hidden_dim, num_classes).to(dev)

    y_train = train_ds.y.numpy().astype(int)
    class_counts = np.bincount(y_train, minlength=num_classes)
    sampler_weights = np.array([1.0 / max(1, class_counts[v]) for v in y_train], dtype=np.float64)
    sampler = WeightedRandomSampler(weights=sampler_weights, num_samples=len(sampler_weights), replacement=True)
    train_loader = DataLoader(train_ds, batch_size=batch_size, sampler=sampler)
    valid_loader = DataLoader(valid_ds, batch_size=batch_size, shuffle=False)

    if loss_name in ('cb_focal', 'cb_focal_ordinal'):
        base_loss = ClassBalancedFocalLoss(class_counts=class_counts, gamma=gamma)
    else:
        weight = torch.tensor((class_counts.sum() / np.maximum(class_counts, 1)) / len(class_counts), dtype=torch.float32, device=dev)
        base_loss = nn.CrossEntropyLoss(weight=weight)
    ord_penalty = OrdinalDistancePenalty(num_classes).to(dev)

    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, epochs=epochs, steps_per_epoch=max(1, len(train_loader))) if use_one_cycle else None

    history_rows = []
    best_metric = -1.0
    best_state = None
    wait = 0

    for epoch in range(1, epochs + 1):
        model.train()
        tr_loss = 0.0
        for seq_x, static_x, y in train_loader:
            seq_x = seq_x.to(dev)
            static_x = static_x.to(dev)
            y = y.to(dev).long()

            logits = model(seq_x, static_x)
            loss = base_loss(logits, y)
            if loss_name == 'cb_focal_ordinal':
                loss = loss + float(ordinal_lambda) * ord_penalty(logits, y)

            opt.zero_grad(set_to_none=True)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            if sched is not None:
                sched.step()
            tr_loss += float(loss.detach().cpu()) * len(y)
        tr_loss /= max(1, len(train_ds))

        valid_metrics = evaluate_sequence_model_plus(model, valid_ds, num_classes=num_classes, device=device)
        monitor = valid_metrics.get('macro_f1', float('nan'))
        history_rows.append({
            'epoch': epoch,
            'train_loss': tr_loss,
            'valid_accuracy': valid_metrics.get('accuracy'),
            'valid_macro_f1': valid_metrics.get('macro_f1'),
            'valid_weighted_f1': valid_metrics.get('weighted_f1'),
            'valid_label_D_f1': valid_metrics.get('label_D_f1'),
            'valid_auc_ovr': valid_metrics.get('auc_ovr'),
        })

        if monitor > best_metric + 1e-6:
            best_metric = monitor
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    return FitResult(
        model=model,
        history=pd.DataFrame(history_rows),
        best_threshold=0.5,
        best_metric_name='macro_f1',
    )


def build_sequence_matrices(df: pd.DataFrame, label_col: str, id_cols: Sequence[str]) -> Tuple[np.ndarray, np.ndarray, np.ndarray, List[str], List[str]]:
    week_map: Dict[str, Dict[int, str]] = {}
    static_cols = []
    for c in df.columns:
        if c in id_cols or c == label_col:
            continue
        if "_week" in c:
            try:
                base, wk = c.rsplit("_week", 1)
                wk = int(wk)
                if 1 <= wk <= 4:
                    week_map.setdefault(base, {})[wk] = c
                    continue
            except Exception:
                pass
        static_cols.append(c)
    seq_bases = sorted([b for b, mp in week_map.items() if all(k in mp for k in [1, 2, 3, 4])])
    seq_x = np.zeros((len(df), 4, len(seq_bases)), dtype=np.float32)
    for j, b in enumerate(seq_bases):
        for w in [1, 2, 3, 4]:
            seq_x[:, w - 1, j] = pd.to_numeric(df[week_map[b][w]], errors="coerce").fillna(0.0).astype(np.float32).values
    static_x = np.zeros((len(df), len(static_cols)), dtype=np.float32)
    for j, c in enumerate(static_cols):
        static_x[:, j] = pd.to_numeric(df[c], errors="coerce").fillna(0.0).astype(np.float32).values
    y = pd.to_numeric(df[label_col], errors="coerce").fillna(-1).astype(int).values
    return seq_x, static_x, y, seq_bases, static_cols
